In [ ]:
def main():
    while True:
        print("\n" + "╔" + "═"*32 + "╗")
        print("║   歡迎使用珍饕餐廳點餐管理系統   ║")
        print("╚" + "═"*32 + "╝")
        print("  1.  ✨ 顧客新點餐開單 (New Order)")
        print("  2.  📊 檢視後台營運報表 (Show Statistics)")
        print("  3.  🗑️  撤銷上一筆錯誤訂單 (Delete Last Order) ※店長權限")
        print("  0.  🚪 關閉並安全退出系統 (Exit)")
        print("-" * 36)

        choice = input(">> 請輸入功能對應數字 (0-3): ").strip()

        if choice == "1":
            current_order = create_order()
            if len(current_order) > 0:
                if checkout(current_order):
                    save_order(current_order)
                    print("✅【系統通知】該筆點餐訂單已成功存檔至後台歷史資料庫。")
            else:
                print("↩️【回主選單】您剛剛未選購任何餐點，已自動取消本次開單。")

        elif choice == "2":
            show_statistics()
        elif choice == "3":
            delete_last_order()
        elif choice == "0":
            print("👋【感謝使用】系統已安全登出並關閉。祝您今日營業額長紅！")
            break
        else:
            print("❌【選項錯誤】無此功能代碼，請重新輸入 0 到 3 之間的整數。")

if __name__ == "__main__":
    main()

# =====================================================================
# [柯盛禾] 二、前端交互介面與中文字寬防呆系統
# =====================================================================

def get_pad_len(string, target_len):
    """【加分功能】動態計算中英文字寬，防止終端機排版破圖"""
    actual_len = sum(2 if ord(c) > 127 else 1 for c in string)
    return max(0, target_len - actual_len)

def show_menu():
    """動態格式化顯示目前餐廳的最新菜單（完美對齊版）"""
    print("\n" + "="*12 + " 經典美味菜單 " + "="*12)
    print(f" {'編號':<4} {'餐點名稱':<14} {'美味價':<6} {'目前庫存':<6}")
    print("-" * 42)
    for key, item in menu.items():
        # 計算需要補多少空格才能對齊
        pad = get_pad_len(item['name'], 16)
        name_str = item['name'] + " " * pad
        print(f"  [{key}]  {name_str} ${item['price']:<7.0f} {item['stock']}份")
    print("=" * 42)

def create_order():
    """引導使用者建立新點餐訂單，具備完整的文字欄位防呆與庫存攔截機制"""
    order = {}
    while True:
        show_menu()
        print("【提示】請輸入想要點購的餐點編號，或輸入 '0' 結束點餐並進入結帳。")
        choice_input = input(">> 請輸入餐點編號: ").strip()

        if not choice_input.isdigit():
            print("❌【輸入錯誤】請勿輸入字母或特殊符號，請重新輸入數字編號！")
            continue

        choice = int(choice_input)
        if choice == 0:
            break

        if choice in menu:
            # 檢查該商品是否還有庫存
            if menu[choice]['stock'] <= 0:
                print(f"❌【已售罄】抱歉，[{menu[choice]['name']}] 已經賣完了！請點選其他餐點。")
                continue

            qty_input = input(f"   請輸入 [{menu[choice]['name']}] 的點購數量: ").strip()

            if not qty_input.isdigit() or int(qty_input) <= 0:
                print("❌【輸入錯誤】數量必須是大於 0 的正整數，請重新輸入。")
                continue

            qty = int(qty_input)

            # 庫存足夠檢查
            already_ordered = order.get(choice, 0)
            if qty + already_ordered > menu[choice]['stock']:
                print(f"❌【庫存不足】[{menu[choice]['name']}] 目前僅剩 {menu[choice]['stock']} 份，您已暫存 {already_ordered} 份，無法再點購 {qty} 份。")
                continue

            if choice in order:
                order[choice] += qty
            else:
                order[choice] = qty
            print(f"✨ 已成功加入暫存訂單: {menu[choice]['name']} x {qty}")
        else:
            print("❌【無此餐點】菜單中沒有這個編號，請看清楚菜單再選一次。")

    return order